# C1 — Expansion + sparsening as locality-sensitive hashing

Companion to [`part2-case-studies/C1`](../part2-case-studies/C1-expansion-and-sparsening.md)
and [`structures/S-11`](../structures/S-11-expanders-and-optimal-degree.md).

We build the insect mushroom body's projection — 50 projection neurons to 2000 Kenyon
cells, each KC sampling ~6 PNs at random, then a winner-take-all that keeps the top 5% —
and check three claims:

1. the resulting binary tag is a **locality-sensitive hash**: tag overlap is a
   predictable function of input correlation, matching $\Phi_2(t,t;\rho)/\alpha$;
2. the expansion **buys linear separability** the PN layer does not have;
3. the in-degree that maximises representation dimension is $K \approx 6$ — the
   anatomical value (Litwin-Kumar et al. 2017).

Pure numpy plus matplotlib. Runs in well under a minute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt, pi

rng = np.random.default_rng(0)
N, M, s, alpha = 50, 2000, 6, 0.05   # PNs, KCs, PNs sampled per KC, WTA fraction

A = np.zeros((M, N))                 # sparse binary random projection
for j in range(M):
    A[j, rng.choice(N, s, replace=False)] = 1.0

print(f'projection {A.shape}, density {A.mean():.3f}, row sums all {int(A.sum(1)[0])}')

## 1. The tag

Three steps, each with an anatomical referent:

| step | code | biology |
|---|---|---|
| expand | `X @ A.T` | PN→KC divergence |
| normalise | z-score across KCs | APL/GGN feedback inhibition |
| sparsify | keep top $\alpha M$ | KC spike threshold |

The normalisation step is not cosmetic — see
[C8 §2.7](../part2-case-studies/C8-divisive-normalization.md): without it a fixed KC
threshold gives either no active cells or all of them.

In [ ]:
def tag(X):
    """Expansion -> normalisation (APL) -> top-k winner-take-all."""
    Y = X @ A.T
    Y = (Y - Y.mean(1, keepdims=True)) / (Y.std(1, keepdims=True) + 1e-12)
    k = int(alpha * M)
    return (Y >= np.partition(Y, -k, axis=1)[:, -k][:, None]).astype(float)

X_demo = np.abs(rng.standard_normal((3, N))) + 0.5
T_demo = tag(X_demo)
print('active KCs per odour:', T_demo.sum(1).astype(int), f'(= {alpha:.0%} of {M})')

## 2. Locality sensitivity

The claim: for standardised KC inputs, two odours with input correlation $\rho$ produce
tags whose expected overlap is

$$\frac{\Pr[X>t,\,Y>t]}{\alpha} = \frac{\Phi_2(t,t;\rho)}{\alpha},
\qquad t = \Phi^{-1}(1-\alpha),$$

with the small-$\alpha$ asymptotic $\alpha^{(1-\rho)/(1+\rho)}$ — the LSH exponent.
Note what this says: overlap depends on $\rho$ *only*, not on the sampling degree $s$ or
on $M$. The hash is scale-free in exactly the way a good LSH should be.

In [ ]:
Phi = np.vectorize(lambda z: 0.5*(1 + erf(z/sqrt(2))))

def Phi2(t, r, n=20000):
    """P(X>t, Y>t) for a standard bivariate normal with correlation r."""
    u = np.linspace(t, t+12, n)
    return float(np.trapezoid(np.exp(-u**2/2)/sqrt(2*pi) *
                              Phi((r*u - t)/sqrt(max(1-r*r, 1e-15))), u))

t_thr = 1.6449                        # Phi^{-1}(1 - 0.05)
rhos, measured, theory, asymp = [], [], [], []

for rho in [0.0, 0.3, 0.5, 0.7, 0.85, 0.95, 0.99]:
    x = rng.standard_normal((400, N)); z = rng.standard_normal((400, N))
    y = np.abs(rho*x + sqrt(1-rho**2)*z) + 0.5
    x = np.abs(x) + 0.5                # non-negative firing rates
    xc, yc = x - x.mean(1, keepdims=True), y - y.mean(1, keepdims=True)
    r = float(np.mean((xc*yc).sum(1) /
              (np.linalg.norm(xc, axis=1)*np.linalg.norm(yc, axis=1))))
    ov = float(((tag(x)*tag(y)).sum(1)/(alpha*M)).mean())
    rhos.append(r); measured.append(ov)
    theory.append(Phi2(t_thr, r)/alpha)
    asymp.append(alpha**((1-r)/(1+r)))

for a, b, c, d in zip(rhos, measured, theory, asymp):
    print(f'  rho={a:6.3f}   overlap={b:6.3f}   Phi2/alpha={c:6.3f}   asymptotic={d:6.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(rhos, measured, 'o-', label='measured tag overlap', lw=2)
ax.plot(rhos, theory, 's--', label=r'$\Phi_2(t,t;\rho)/\alpha$')
ax.plot(rhos, asymp, '^:', label=r'$\alpha^{(1-\rho)/(1+\rho)}$')
ax.set_xlabel('input correlation  $\\rho$'); ax.set_ylabel('tag overlap')
ax.set_title('The mushroom body tag is locality-sensitive')
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

The exact expression tracks the measurement closely; the asymptotic runs high because
$\alpha=0.05$ is not small enough for it. **Try:** drop `alpha` to 0.01 and watch the
asymptotic converge onto the exact curve.

## 3. What the expansion buys: separability

Cover's theorem says a linear readout of $N=50$ PNs can separate at most $\sim 2N = 100$
random dichotomies. The KC layer should do far better — despite carrying no new
information, since it is a deterministic function of the PN layer.

In [ ]:
def sep_frac(F, P, trials=20):
    """Fraction of random balanced dichotomies of P points that are linearly separable."""
    F = np.hstack([F[:P], np.ones((P, 1))]); ok = 0
    for _ in range(trials):
        lab = rng.choice([-1., 1.], P)
        w = np.linalg.lstsq(F, lab, rcond=None)[0]
        for _ in range(400):
            bad = lab*(F@w) < 1e-9
            if not bad.any(): break
            w += 0.05*(F[bad]*lab[bad, None]).mean(0)
        ok += bool((lab*(F@w) > 0).all())
    return ok/trials

X = np.abs(rng.standard_normal((600, N))); KC = tag(X)
Ps = [40, 80, 150, 300, 600]
pn = [sep_frac(X, P) for P in Ps]
kc = [sep_frac(KC, P) for P in Ps]
for P, a, b in zip(Ps, pn, kc):
    print(f'  P={P:4d}   PN layer {a:.2f}   KC layer {b:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(Ps, pn, 'o-', label=f'PN layer (N={N})', lw=2)
ax.plot(Ps, kc, 's-', label=f'KC layer (M={M})', lw=2)
ax.axvline(2*N, color='k', ls=':', label='Cover bound $2N$ for PNs')
ax.set_xlabel('number of odours $P$'); ax.set_ylabel('fraction separable')
ax.set_title('Expansion buys separability'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

The PN layer collapses right where Cover says it must. The KC layer does not — and it
contains no information the PN layer lacked. **That is the whole point of an expansion:
it trades neurons for readout simplicity, not for information.**

## 4. Is there an optimal in-degree?

Now the quantitative claim from [S-11](../structures/S-11-expanders-and-optimal-degree.md).
Two pressures oppose each other:

- $K$ too small — each KC copies one or two PNs; little mixing, few new dimensions.
- $K$ too large — KCs share inputs, become correlated, and dimension collapses.

Measure dimension by participation ratio
$\mathrm{PR} = (\sum_i\lambda_i)^2/\sum_i\lambda_i^2$ and sweep $K$.

**Read the result carefully — this cell does not reproduce the famous number, and the
reason is instructive.**

In [ ]:
def pr_for_degree(K, P=500, n_pn_modes=8, sparsity=0.05, seed=1):
    r = np.random.default_rng(seed)
    L = r.standard_normal((N, n_pn_modes))
    Sigma = L@L.T + 0.5*np.eye(N)                  # correlated PN activity
    Y = r.multivariate_normal(np.zeros(N), Sigma, size=P).T   # N x P
    Ak = np.zeros((M, N))
    for j in range(M):
        Ak[j, r.choice(N, K, replace=False)] = 1.0
    U = Ak @ Y
    thr = np.quantile(U, 1-sparsity, axis=1, keepdims=True)
    R = np.maximum(U - thr, 0)
    ev = np.linalg.eigvalsh(np.cov(R))
    return ev.sum()**2 / (ev**2).sum()

Ks = [1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 16, 20, 28, 40, 50]
PRs = [np.mean([pr_for_degree(K, seed=sd) for sd in (1, 2, 3)]) for K in Ks]
best = Ks[int(np.argmax(PRs))]
for K, v in zip(Ks, PRs):
    print(f'  K={K:3d}   PR={v:7.2f}' + ('   <-- max' if K == best else ''))
print(f'\nthis toy peaks at K = {best};  Drosophila anatomy is ~6')

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(Ks, PRs, 'o-', lw=2)
ax.axvline(6, color='crimson', ls='--', label='anatomy: ~6 claws per KC')
ax.axvline(best, color='steelblue', ls=':', label=f'this toy: K={best}')
ax.set_xscale('log'); ax.set_xlabel('KC in-degree $K$')
ax.set_ylabel('participation ratio of KC layer')
ax.set_title('An interior optimum exists — but its location is not universal')
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

### What just happened, and why it is worth more than a rigged demo

The **qualitative** claim survives: dimension is maximised at an *interior* $K$, falling
off in both directions. That is the non-trivial part, and it is what makes "why six?" a
well-posed question rather than a curiosity.

The **quantitative** claim does not reproduce here: this toy peaks around $K=2$–$3$, not
$6$. That is not a bug in the theory, it is a difference in the model. What this cell is
missing relative to Litwin-Kumar et al. (2017):

- **A noise model.** Here the PN input is noiseless, so larger $K$ has no upside beyond
  mixing. In the real calculation, sampling more PNs *averages away input noise*, which is
  a genuine pressure toward larger $K$. Add per-trial noise to `Y` and watch the optimum
  move right — it does, though not all the way to 6.
- **The fly's actual PN correlation structure**, rather than an 8-mode Gaussian invention.
- **The objective.** They optimise dimension in a setting tied to downstream learning
  performance, not raw PR of a thresholded response.

Keep this as a worked instance of [S2](../part3-synthesis/S2-degeneracy-and-limits.md)'s
point: a prediction of a *specific number* is only as good as the assumptions feeding it,
and "my toy reproduced the number" is a claim you should distrust in your own work as much
as in anyone else's. Reproducing $K\approx6$ properly is the exercise below.

## Things to try

1. **Move the optimum.** Change `n_pn_modes` (how correlated the PN activity is) and
   `sparsity`. The optimum shifts — which is the honest caveat on the "6": it is optimal
   *for the fly's measured PN statistics and KC sparsity*, not universally.
2. **Break the normalisation.** Delete the z-scoring line in `tag` and rerun §2 with
   odours at different overall intensities. Locality sensitivity should degrade — the tag
   starts encoding concentration instead of identity.
3. **Non-random connectivity.** Replace the random `A` with a structured one (e.g. each KC
   reads 6 *adjacent* PNs). Does tag overlap still follow the theory curve? Why not?
4. **Exercise C1.5** in the case study asks you to derive the Babadi–Sompolinsky optimum
   analytically; this notebook gives you the numerics to check it against.